## 0. Install


In [ ]:
!pip install -q llama-index-llms-ollama

In [ ]:
!pip install -q \
    llama-parse \
    llama-index-core \
    llama-index-embeddings-huggingface \
    llama-index-llms-huggingface \
    llama-index-vector-stores-chroma \
    chromadb \
    langchain \
    rank-bm25 \
    flashrank \
    sentence-transformers \
    transformers \
    accelerate \
    torch \
    pydantic \
    pypdf \
    numpy

print("Dependencies assumed installed. Uncomment for a fresh environment.")

## 1. Imports & logging


In [ ]:
import os
import json
import time
import hashlib
import logging
import pickle
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Tuple, Optional, Any, Iterable, Union
from collections import defaultdict

import numpy as np
from pydantic import BaseModel, Field, field_validator

# ----- Logging: structured, timestamped, single source of truth -----
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)-22s | %(message)s",
    datefmt="%H:%M:%S",
)
for noisy in ("httpx", "httpcore", "urllib3", "sentence_transformers", "transformers"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

log = logging.getLogger("rag.main")
log.info("Logging configured.")
from dotenv import load_dotenv
load_dotenv()

## 2. Config (Pydantic)


In [ ]:
class RAGConfig(BaseModel):
    """Single source of truth for every tunable in the pipeline."""

    # Paths
    pdf_path: str = "./data/NIST.CSWP.29.pdf"
    persist_dir: str = "./persist"
    chroma_path: str = "./persist/chroma_db"
    chroma_collection: str = "nist_csf"

    # Chunking
    chunk_size: int = 800
    chunk_overlap: int = 150
    min_chunk_chars: int = 60

    # Embeddings (local, free)
    embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"
    embedding_dim: int = 384
    multilingual_model: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    multilingual_dim: int = 384

    # open ai llms 
    qa_model: str = "gpt-4o-mini"
    rag_model: str = "gpt-4o-mini"

    # Retrieval
    bm25_top_k: int = 15
    faiss_top_k: int = 15
    rrf_k: int = 60
    final_top_k: int = 5

    # Reranker
    reranker_model: str = "ms-marco-MiniLM-L-12-v2"

    # Semantic cache
    cache_similarity_threshold: float = 0.85
    cache_max_entries: int = 256

    
    metadata_section_boost: float = 0.01
    metadata_subcategory_boost: float = 0.02
    metadata_page_boost: float = 0.01

    questions_per_chunk: int = 1
    seed: int = 42

    @field_validator("chunk_overlap")
    @classmethod
    def overlap_must_be_less_than_size(cls, v, info):
        if "chunk_size" in info.data and v >= info.data["chunk_size"]:
            raise ValueError("chunk_overlap must be < chunk_size")
        return v


CONFIG = RAGConfig()
Path(CONFIG.persist_dir).mkdir(parents=True, exist_ok=True)
Path(CONFIG.chroma_path).mkdir(parents=True, exist_ok=True)
np.random.seed(CONFIG.seed)

log.info("Config loaded:")
for k, v in CONFIG.model_dump().items():
    log.info(f"  {k:32s} = {v}")

## 3. PDF Parsing — LlamaParse with pypdf fallback


In [ ]:
class ParsedDocument(BaseModel):
    """Output schema both parsers conform to."""
    text: str
    page_number: int
    source_file: str
    parser: str  # 'llamaparse' or 'pypdf-fallback'


def _coerce_page(label, fallback: int) -> int:
   
    if label is None:
        return fallback
    if isinstance(label, int):
        return label
    s = str(label).strip()
    if s.isdigit():
        return int(s)
    # '3-4' → take the first numeric component
    head = s.split("-")[0].split(",")[0].strip()
    if head.isdigit():
        return int(head)
    # Roman numerals or anything else → keep ordinal fallback
    return fallback


def parse_with_llamaparse(pdf_path: str) -> List[ParsedDocument]:
    """Top-tier parsing path. Requires LLAMA_CLOUD_API_KEY env var."""
    api_key = os.getenv("LLAMA_CLOUD_API_KEY")
    if not api_key:
        raise RuntimeError("LLAMA_CLOUD_API_KEY not set")

    from llama_parse import LlamaParse
    parser = LlamaParse(
        api_key=api_key,
        result_type="markdown",
        verbose=False,
        language="en",
    )
    log.info("Parsing with LlamaParse (markdown mode)…")
    docs = parser.load_data(pdf_path)
    out = []
    for i, d in enumerate(docs):
        page_label = d.metadata.get("page_label") if hasattr(d, "metadata") else None
        out.append(ParsedDocument(
            text=d.text,
            page_number=_coerce_page(page_label, i + 1),
            source_file=Path(pdf_path).name,
            parser="llamaparse",
        ))
    log.info(f"LlamaParse produced {len(out)} page-documents.")
    return out


def parse_with_pypdf(pdf_path: str) -> List[ParsedDocument]:
    """Local fallback. Always works — no network, no API key."""
    from pypdf import PdfReader
    log.info("Parsing with pypdf fallback…")
    reader = PdfReader(pdf_path)
    out = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        text = " ".join(text.split())
        if not text.strip():
            continue
        out.append(ParsedDocument(
            text=text,
            page_number=i + 1,
            source_file=Path(pdf_path).name,
            parser="pypdf-fallback",
        ))
    log.info(f"pypdf produced {len(out)} page-documents.")
    return out


def parse_pdf(pdf_path: str) -> List[ParsedDocument]:
    """Try LlamaParse first; fall back gracefully."""
    try:
        return parse_with_llamaparse(pdf_path)
    except Exception as e:
        log.warning(f"LlamaParse unavailable ({type(e).__name__}: {e}). Falling back to pypdf.")
        return parse_with_pypdf(pdf_path)


t0 = time.perf_counter()
parsed_docs = parse_pdf(CONFIG.pdf_path)
log.info(f"Parsing complete in {time.perf_counter() - t0:.2f}s — {len(parsed_docs)} pages.")
log.info(f"First page preview: {parsed_docs[0].text[:200]}…")

## 4. Domain-aware metadata extraction


In [ ]:
import re

CSF_FUNCTIONS = ["GOVERN", "IDENTIFY", "PROTECT", "DETECT", "RESPOND", "RECOVER"]
FUNCTION_PATTERN = re.compile(
    r"\b(" + "|".join(CSF_FUNCTIONS) + r")\s*\(?(GV|ID|PR|DE|RS|RC)?\)?\b",
    re.IGNORECASE,
)
SUBCATEGORY_PATTERN = re.compile(r"\b(GV|ID|PR|DE|RS|RC)\.[A-Z]{2}-\d{2}\b")


def infer_section(text: str) -> str:
    counts = defaultdict(int)
    for match in FUNCTION_PATTERN.finditer(text):
        fn = match.group(1).upper()
        if fn in CSF_FUNCTIONS:
            counts[fn] += 1
    if not counts:
        return "GENERAL"
    return max(counts.items(), key=lambda kv: kv[1])[0]


def extract_subcategories(text: str) -> List[str]:
    return sorted(set(m.group(0) for m in SUBCATEGORY_PATTERN.finditer(text)))


sample_text = "GV.SC-07: The risks posed by a supplier are understood. Under GOVERN, this is critical."
print("Inferred section:", infer_section(sample_text))
print("Subcategories found:", extract_subcategories(sample_text))

## 5. Chunking


In [ ]:
class Chunk(BaseModel):
    chunk_id: str
    text: str
    source_file: str
    page_number: int
    section: str
    subcategories: List[str] = Field(default_factory=list)
    language: str = "en"
    document_title: str = "NIST Cybersecurity Framework (CSF) 2.0"
    char_start: int = 0
    char_end: int = 0
    parser: str = "unknown"
    summary: Optional[str] = None
    synthetic_questions: List[str] = Field(default_factory=list)


def make_chunk_id(text: str, page: int, position: int) -> str:
    h = hashlib.sha1(f"{page}|{position}|{text[:200]}".encode("utf-8")).hexdigest()
    return f"chunk_{h[:12]}"


def chunk_documents(
    docs: List[ParsedDocument],
    size: int = CONFIG.chunk_size,
    overlap: int = CONFIG.chunk_overlap,
    min_chars: int = CONFIG.min_chunk_chars,
) -> List[Chunk]:
    chunks: List[Chunk] = []
    for doc in docs:
        text = doc.text
        if len(text) < min_chars:
            continue
        start = 0
        position = 0
        while start < len(text):
            end = min(start + size, len(text))
            if end < len(text):
                next_break = -1
                for sep in ["\n\n", ". ", "\n", "; "]:
                    cand = text.find(sep, end, min(end + 80, len(text)))
                    if cand != -1:
                        next_break = cand + len(sep)
                        break
                if next_break != -1:
                    end = next_break
            window = text[start:end].strip()
            if len(window) >= min_chars:
                cid = make_chunk_id(window, doc.page_number, position)
                chunks.append(Chunk(
                    chunk_id=cid,
                    text=window,
                    source_file=doc.source_file,
                    page_number=doc.page_number,
                    section=infer_section(window),
                    subcategories=extract_subcategories(window),
                    char_start=start,
                    char_end=end,
                    parser=doc.parser,
                ))
            if end == len(text):
                break
            start = end - overlap
            position += 1
    return chunks


t0 = time.perf_counter()
chunks = chunk_documents(parsed_docs)
log.info(f"Chunked {len(parsed_docs)} pages → {len(chunks)} chunks in {time.perf_counter() - t0:.2f}s")

section_counts = defaultdict(int)
for c in chunks:
    section_counts[c.section] += 1
log.info("Chunks by inferred CSF Function:")
for fn, n in sorted(section_counts.items(), key=lambda x: -x[1]):
    log.info(f"  {fn:10s} {n:4d}")

In [ ]:
print("=" * 80)
print("SAMPLE CHUNKS")
print("=" * 80)
for c in chunks[:3]:
    print(f"\n[{c.chunk_id}] page={c.page_number} section={c.section} subcats={c.subcategories}")
    print(f"  {c.text[:300]}…")
print(f"\nTotal chunks: {len(chunks)}")

### 5.3 Synthetic question generation




In [ ]:
from llama_index.llms.openai import OpenAI

log.info(f"Loading model: {CONFIG.qa_model} (OpenAI) for chunk enrichment…")
_t0 = time.perf_counter()
qa_llm = OpenAI(
    model=CONFIG.qa_model,
    temperature=0.1,
    max_tokens=256,
    timeout=60.0,
)
log.info(f"{CONFIG.qa_model} ready in {time.perf_counter()-_t0:.2f}s")


def llm_generate_questions(passage: str, n: int = CONFIG.questions_per_chunk) -> List[str]:
    """Generate retrieval-style questions this chunk answers."""
    prompt = f"""
Generate {n} short retrieval questions for this cybersecurity passage.

Rules:
- One question per line
- No numbering
- No explanations
- Preserve identifiers exactly (GV.SC-07, PR.AA-01, etc.)
- Use cybersecurity terminology from the passage
- Questions should improve RAG retrieval quality
- Keep questions concise and specific

Passage:
{passage[:1200]}

Questions:
"""
    try:
        response = qa_llm.complete(prompt)
        gen = str(response)
    except Exception as e:
        log.warning(f"qa_llm failed on a chunk ({e}); using fallback question.")
        return ["What cybersecurity guidance does this passage provide?"]

    questions = [
        ln.strip().lstrip("0123456789.-) ").strip()
        for ln in gen.split("\n")
        if ln.strip()
    ]
    questions = [q for q in questions if len(q) > 10]
    return questions[:n] if questions else [
        "What cybersecurity guidance does this passage provide?"
    ]


_corpus_sig_for_questions = hashlib.md5(
    ("|".join(c.chunk_id for c in chunks) + f"|{CONFIG.qa_model}|{CONFIG.questions_per_chunk}").encode()
).hexdigest()[:12]
_questions_cache_path = Path(CONFIG.persist_dir) / f"synthetic_questions_{_corpus_sig_for_questions}.json"

if _questions_cache_path.exists():
    log.info(f"Loading cached synthetic questions from {_questions_cache_path}")
    cached_qs = json.loads(_questions_cache_path.read_text())
    for c in chunks:
        c.synthetic_questions = cached_qs.get(c.chunk_id, [])
else:
    log.info(f"Generating synthetic questions for {len(chunks)} chunks (one-time cost)…")
    t0 = time.perf_counter()
    for i, c in enumerate(chunks):
        c.synthetic_questions = llm_generate_questions(c.text)
        if (i + 1) % 25 == 0:
            log.info(f"  enriched {i+1}/{len(chunks)} chunks…")
    elapsed = time.perf_counter() - t0
    log.info(f"Enrichment complete in {elapsed:.1f}s ({elapsed/len(chunks):.2f}s/chunk avg)")
    _questions_cache_path.write_text(
        json.dumps({c.chunk_id: c.synthetic_questions for c in chunks}, ensure_ascii=False)
    )
    log.info(f"Persisted to {_questions_cache_path}")

# Sanity check: how many chunks actually got questions?
_with_qs = sum(1 for c in chunks if c.synthetic_questions)
log.info(f"Chunks with synthetic questions: {_with_qs}/{len(chunks)} "
         f"({100*_with_qs/len(chunks):.1f}%)")



### 5.4 The fused embedding string

A single helper joins chunk text and synthetic questions into one input. **Both BM25 and the embedder will use this same string** — keeping retrievers aligned on what they're indexing.


## 6. Embedding helpers


In [ ]:
from sentence_transformers import SentenceTransformer

_embed_model_cache: Dict[str, SentenceTransformer] = {}

def get_embedding_model(model_name: str) -> SentenceTransformer:
    if model_name not in _embed_model_cache:
        log.info(f"Loading embedding model: {model_name}")
        t0 = time.perf_counter()
        _embed_model_cache[model_name] = SentenceTransformer(model_name)
        log.info(f"  loaded in {time.perf_counter() - t0:.2f}s")
    return _embed_model_cache[model_name]


def embed_texts(texts: List[str], model_name: str = CONFIG.embedding_model) -> np.ndarray:
    """Returns L2-normalized float32 matrix."""
    model = get_embedding_model(model_name)
    vecs = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    return vecs

In [ ]:
class Chunk(BaseModel):
    chunk_id: str
    text: str
    source_file: str
    page_number: int
    section: str
    subcategories: List[str] = Field(default_factory=list)
    language: str = "en"
    document_title: str = "NIST Cybersecurity Framework (CSF) 2.0"
    char_start: int = 0
    char_end: int = 0
    parser: str = "unknown"
    summary: Optional[str] = None
    synthetic_questions: List[str] = Field(default_factory=list)


def make_chunk_id(text: str, page: int, position: int) -> str:
    h = hashlib.sha1(f"{page}|{position}|{text[:200]}".encode("utf-8")).hexdigest()
    return f"chunk_{h[:12]}"


def build_embedding_string(chunk: Chunk) -> str:
    """Fuse text + summary + synthetic questions into one embedding-time representation."""
    parts = [chunk.text]
    if chunk.summary:
        parts.append(f"Summary: {chunk.summary}")
    if chunk.synthetic_questions:
        parts.append("Anticipated questions: " + " | ".join(chunk.synthetic_questions))
    return " ||| ".join(parts)


def chunk_documents(
    docs: List[ParsedDocument],
    size: int = CONFIG.chunk_size,
    overlap: int = CONFIG.chunk_overlap,
    min_chars: int = CONFIG.min_chunk_chars,
) -> List[Chunk]:
    chunks: List[Chunk] = []
    for doc in docs:
        text = doc.text
        if len(text) < min_chars:
            continue
        start = 0
        position = 0
        while start < len(text):
            end = min(start + size, len(text))
            if end < len(text):
                next_break = -1
                for sep in ["\n\n", ". ", "\n", "; "]:
                    cand = text.find(sep, end, min(end + 80, len(text)))
                    if cand != -1:
                        next_break = cand + len(sep)
                        break
                if next_break != -1:
                    end = next_break
            window = text[start:end].strip()
            if len(window) >= min_chars:
                cid = make_chunk_id(window, doc.page_number, position)
                chunks.append(Chunk(
                    chunk_id=cid,
                    text=window,
                    source_file=doc.source_file,
                    page_number=doc.page_number,
                    section=infer_section(window),
                    subcategories=extract_subcategories(window),
                    char_start=start,
                    char_end=end,
                    parser=doc.parser,
                ))
            if end == len(text):
                break
            start = end - overlap
            position += 1
    return chunks


t0 = time.perf_counter()
chunks = chunk_documents(parsed_docs)
log.info(f"Chunked {len(parsed_docs)} pages → {len(chunks)} chunks in {time.perf_counter() - t0:.2f}s")

section_counts = defaultdict(int)
for c in chunks:
    section_counts[c.section] += 1
log.info("Chunks by inferred CSF Function:")
for fn, n in sorted(section_counts.items(), key=lambda x: -x[1]):
    log.info(f"  {fn:10s} {n:4d}")

## 7. Vector store (Chroma) — content-hash versioned



In [ ]:
import chromadb
from llama_index.core import VectorStoreIndex
from llama_index.core.schema import MetadataMode, TextNode
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

DOC_TITLE = "NIST Cybersecurity Framework (CSF) 2.0"

EMBED_EXCLUDE = ["chunk_id", "file_name", "page_number", "parser",
                 "char_start", "char_end", "language", "raw_text",
                 "synthetic_questions", "retrieval_text"]
LLM_EXCLUDE = ["chunk_id", "file_name", "page_number", "parser",
               "char_start", "char_end", "language", "raw_text",
               "synthetic_questions", "retrieval_text"]


def chunk_to_node(c: Chunk) -> TextNode:
   
    fused = build_embedding_string(c)
    return TextNode(
        text=fused,                    
        id_=c.chunk_id,
        metadata={
            "document_title": DOC_TITLE,
            "section": c.section,
            "subcategories": ", ".join(c.subcategories),
            "chunk_id": c.chunk_id,
            "file_name": c.source_file,
            "page_number": c.page_number,
            "parser": c.parser,
            "language": c.language,
            "raw_text": c.text,      
            "synthetic_questions": "\n".join(c.synthetic_questions),
        },
        excluded_embed_metadata_keys=EMBED_EXCLUDE,
        excluded_llm_metadata_keys=LLM_EXCLUDE,
        metadata_separator="\n",        
        metadata_template="{key}: {value}",
        text_template="Metadata:\n{metadata_str}\n---\nContent:\n{content}",
    )


nodes = [chunk_to_node(c) for c in chunks]

# Verify exclusions
print("=" * 80)
print("WHAT THE EMBEDDER SEES (section + title only as metadata; content is fused):")
print("=" * 80)
print(nodes[len(nodes) // 2].get_content(metadata_mode=MetadataMode.EMBED)[:600], "…")

In [ ]:

def compute_corpus_hash(nodes_, embed_model_name: str) -> str:
    """Stable hash over the actual content + embedding model. Changes when ANY
    of these change: chunk text, fusion (questions), node count, embed model."""
    h = hashlib.sha256()
    h.update(embed_model_name.encode())
    h.update(f"|n={len(nodes_)}".encode())
    for n in nodes_:
        h.update(b"|")
        h.update(n.id_.encode())
        h.update(b":")
        h.update(n.text.encode("utf-8"))
    return h.hexdigest()[:16]


primary_hash = compute_corpus_hash(nodes, CONFIG.embedding_model)
log.info(f"Primary corpus hash: {primary_hash}")

hf_embeddings = HuggingFaceEmbedding(model_name=CONFIG.embedding_model)
db = chromadb.PersistentClient(path=CONFIG.chroma_path)


def get_or_build_collection(name: str, want_hash: str, nodes_, embed_model):
    """Load if hash matches, otherwise rebuild. Stores hash in collection metadata."""
    existing = None
    try:
        existing = db.get_collection(name)
    except Exception:
        existing = None

    if existing is not None:
        meta = existing.metadata or {}
        stored_hash = meta.get("corpus_hash")
        if stored_hash == want_hash and existing.count() == len(nodes_):
            log.info(f"[{name}] hash match ({stored_hash}); loading existing.")
            vs = ChromaVectorStore(chroma_collection=existing)
            idx = VectorStoreIndex.from_vector_store(vector_store=vs, embed_model=embed_model)
            return existing, vs, idx
        log.info(f"[{name}] hash mismatch (stored={stored_hash}, want={want_hash}) — rebuilding.")
        db.delete_collection(name)

    coll = db.create_collection(name=name, metadata={"corpus_hash": want_hash})
    vs = ChromaVectorStore(chroma_collection=coll)
    sc = StorageContext.from_defaults(vector_store=vs)
    log.info(f"[{name}] building index from {len(nodes_)} nodes…")
    t0 = time.perf_counter()
    idx = VectorStoreIndex(nodes_, storage_context=sc, embed_model=embed_model)
    log.info(f"[{name}] built and persisted in {time.perf_counter()-t0:.1f}s ({coll.count()} vectors)")
    return coll, vs, idx


chroma_collection, vector_store, index = get_or_build_collection(
    CONFIG.chroma_collection, primary_hash, nodes, hf_embeddings,
)

chroma_retriever = index.as_retriever(similarity_top_k=CONFIG.faiss_top_k)


def chroma_search(query: str, k: int):
    return [(nws.node, float(nws.score)) for nws in chroma_retriever.retrieve(query)[:k]]


log.info(f"Chroma vector store ready: {chroma_collection.count()} nodes.")

In [ ]:
# Same nodes, multilingual embedder, separate Chroma collection — content-hash versioned.
multi_embed = HuggingFaceEmbedding(model_name=CONFIG.multilingual_model)
multi_hash = compute_corpus_hash(nodes, CONFIG.multilingual_model)
log.info(f"Multilingual corpus hash: {multi_hash}")

multi_collection, multi_vector_store, multi_index = get_or_build_collection(
    CONFIG.chroma_collection + "_multi", multi_hash, nodes, multi_embed,
)

multi_retriever = multi_index.as_retriever(similarity_top_k=CONFIG.faiss_top_k)


def chroma_search_multi(query: str, k: int):
    return [(nws.node, float(nws.score)) for nws in multi_retriever.retrieve(query)[:k]]


log.info(f"Multilingual vector store ready: {multi_collection.count()} nodes.")

## 8. BM25




In [ ]:
from rank_bm25 import BM25Okapi
import re
from typing import List

_IDENTIFIER_RE = re.compile(r"\b([A-Z]{2})\.([A-Z]{2})-(\d{2})\b")
_WORD_RE = re.compile(r"[A-Za-z0-9]+(?:[-_][A-Za-z0-9]+)*")


def _normalize_identifier(m):
    return f"{m.group(1).lower()}_{m.group(2).lower()}_{m.group(3)}"


def cyber_tokenize(text: str) -> List[str]:
    # Convert identifiers into stable BM25-safe tokens
    normalized = _IDENTIFIER_RE.sub(_normalize_identifier, text)

    # Extract tokens
    tokens = [t.lower() for t in _WORD_RE.findall(normalized)]

    return tokens


print(cyber_tokenize("Subcategory GV.SC-07 covers supplier risks."))

In [ ]:
class BM25Retriever:
    
    
    def __init__(self, chunks_: List[Chunk], nodes_: List[TextNode]):
        assert len(chunks_) == len(nodes_)
        self.chunks = chunks_
        self.nodes = nodes_   # parallel; same order
        self.log = logging.getLogger("rag.bm25")
        self.log.info(f"Tokenizing {len(chunks_)} chunks for BM25 (clean fused string)…")
        t0 = time.perf_counter()
        # Use the SAME fused string the embedder sees.
        self.tokenized = [cyber_tokenize(build_embedding_string(c)) for c in chunks_]
        self.bm25 = BM25Okapi(self.tokenized, k1=1.5, b=0.75)
        self.log.info(f"BM25 ready in {time.perf_counter()-t0:.2f}s")

    def search(self, query: str, k: int):
        scores = self.bm25.get_scores(cyber_tokenize(query))
        top = np.argpartition(scores, -k)[-k:]
        top = top[np.argsort(-scores[top])]
        return [(self.nodes[i], float(scores[i])) for i in top if scores[i] > 0]


bm25_retriever = BM25Retriever(chunks, nodes)

results = bm25_retriever.search("GV.SC-07", k=3)
print(f"\nBM25 results for query='GV.SC-07':")
for n, s in results:
    print(f"  score={s:.3f}  page={n.metadata.get('page_number')}  "
          f"section={n.metadata.get('section')}")
    print(f"    {n.metadata.get('raw_text', n.text)[:140]}…")

## 9. Semantic cache



In [ ]:
class SemanticCache:
   
    def __init__(
        self,
        threshold: float,
        max_entries: int,
        signature: str,
        en_model_name: str,
        multi_model_name: str,
    ):
        self.threshold = threshold
        self.max_entries = max_entries
        self.signature = signature
        self.en_model_name = en_model_name
        self.multi_model_name = multi_model_name

        self.embeddings: List[np.ndarray] = []
        self.queries: List[str] = []
        self.answers: List[Dict[str, Any]] = []
        self.embed_models: List[str] = []         
        self.last_access: List[int] = []        
        self._clock = 0

        self.hits = 0
        self.misses = 0
        self.log = logging.getLogger("rag.cache")

    def _tick(self) -> int:
        self._clock += 1
        return self._clock

    def _model_for_query(self, query: str) -> str:

        return self.multi_model_name if detect_language(query) == "ar" else self.en_model_name

    def _embed(self, text: str, model_name: str) -> np.ndarray:
        return embed_texts([text], model_name)[0]

    def lookup(self, query: str) -> Optional[Dict[str, Any]]:
        if not self.embeddings:
            self.misses += 1
            self.log.info(f"CACHE MISS (empty cache) | query={query[:60]!r}")
            return None

        model_name = self._model_for_query(query)
        q_vec = self._embed(query, model_name)

       
        candidate_idx = [i for i, m in enumerate(self.embed_models) if m == model_name]
        if not candidate_idx:
            self.misses += 1
            self.log.info(
                f"CACHE MISS (no entries under model={model_name}) | "
                f"query={query[:60]!r}"
            )
            return None

        emb_matrix = np.vstack([self.embeddings[i] for i in candidate_idx])
        sims = emb_matrix @ q_vec 
        best_local = int(np.argmax(sims))
        best_idx = candidate_idx[best_local]
        best_sim = float(sims[best_local])

        if best_sim >= self.threshold:
            self.hits += 1
            self.last_access[best_idx] = self._tick()
            self.log.info(
                f"CACHE HIT | sim={best_sim:.4f} | thr={self.threshold} | "
                f"model={model_name} | matched={self.queries[best_idx][:60]!r}"
            )
            return {
                **self.answers[best_idx],
                "_cache_hit": True,
                "_cache_similarity": best_sim,
                "_cache_matched_query": self.queries[best_idx],
            }

        self.misses += 1
        self.log.info(
            f"CACHE MISS | best_sim={best_sim:.4f} < thr={self.threshold} | model={model_name}"
        )
        return None

    def store(self, query: str, answer: Dict[str, Any]):
        
        if len(self.embeddings) >= self.max_entries:
            evict_idx = int(np.argmin(self.last_access))
            self.log.info(f"CACHE EVICT (LRU) | query={self.queries[evict_idx][:60]!r}")
            for arr in (self.embeddings, self.queries, self.answers,
                        self.embed_models, self.last_access):
                del arr[evict_idx]

        model_name = self._model_for_query(query)
        self.embeddings.append(self._embed(query, model_name))
        self.queries.append(query)
        self.answers.append(answer)
        self.embed_models.append(model_name)
        self.last_access.append(self._tick())
        self.log.info(f"CACHE STORE | size={len(self.embeddings)}/{self.max_entries} | model={model_name}")

    def stats(self) -> Dict[str, Any]:
        total = self.hits + self.misses
        return {
            "size": len(self.embeddings),
            "hits": self.hits,
            "misses": self.misses,
            "hit_rate": self.hits / total if total else 0.0,
        }


def detect_language(text: str) -> str:
    return "ar" if sum(1 for c in text if "\u0600" <= c <= "\u06FF") > 3 else "en"


corpus_signature = hashlib.md5(
    "".join(c.text for c in chunks).encode()
).hexdigest()

semantic_cache = SemanticCache(
    threshold=CONFIG.cache_similarity_threshold,
    max_entries=CONFIG.cache_max_entries,
    signature=corpus_signature,
    en_model_name=CONFIG.embedding_model,
    multi_model_name=CONFIG.multilingual_model,
)

log.info(f"Semantic cache initialized | signature={corpus_signature[:12]}…")
print("\nSemantic CAG layer ready.")
print("Cache stats:", semantic_cache.stats())

## 10. Reciprocal Rank Fusion


In [ ]:
def reciprocal_rank_fusion(rank_lists, k: int = CONFIG.rrf_k):
    """RRF over (item, score) lists. Items must expose .metadata['chunk_id']."""
    aggregated = {}
    for retriever_idx, rank_list in enumerate(rank_lists):
        for rank, (item, score) in enumerate(rank_list, start=1):
            node = getattr(item, "node", item)
            chunk_id = node.metadata["chunk_id"]
            entry = aggregated.setdefault(
                chunk_id,
                {"node": node, "rrf_score": 0.0, "ranks": {}, "scores": {}},
            )
            entry["rrf_score"] += 1.0 / (k + rank)
            entry["ranks"][f"retriever_{retriever_idx}"] = rank
            entry["scores"][f"retriever_{retriever_idx}"] = score

    fused = sorted(aggregated.values(), key=lambda e: -e["rrf_score"])
    return [
        (e["node"], e["rrf_score"], {"ranks": e["ranks"], "scores": e["scores"]})
        for e in fused
    ]


q = "supply chain risk management"
bm25_results = bm25_retriever.search(q, k=10)
faiss_results = chroma_search(q, k=10)
fused = reciprocal_rank_fusion([bm25_results, faiss_results])

print(f"\nFusion demo for query={q!r}")
print(f"  BM25={len(bm25_results)} | Semantic={len(faiss_results)} | Fused={len(fused)}")
print("\nTop-5 after RRF:")
for node, rrf, dbg in fused[:5]:
    print(f"\nrrf={rrf:.5f}  page={node.metadata['page_number']}  "
          f"section={node.metadata['section']:8s}  ranks={dbg['ranks']}")
    print("-" * 100)
    print(f"{node.metadata.get('raw_text', node.text)[:250]}…")

## 11. Metadata-aware soft reranking




In [ ]:
def infer_filters_from_query(query: str) -> Dict[str, Any]:
    filters: Dict[str, Any] = {}
    q_upper = query.upper()
    fns_in_query = [fn for fn in CSF_FUNCTIONS if re.search(r"\b" + fn + r"\b", q_upper)]
    if fns_in_query:
        filters["sections"] = fns_in_query
    subcats = [m.group(0) for m in SUBCATEGORY_PATTERN.finditer(query)]
    if subcats:
        filters["subcategories"] = subcats
    page_match = re.search(r"\bpage\s+(\d+)\b", query, re.IGNORECASE)
    if page_match:
        filters["page"] = int(page_match.group(1))
    return filters


def apply_metadata_filters(fused, filters):
   
    if not filters:
        return fused

    boosted = []
    for node, rrf, dbg in fused:
        metadata = node.metadata
        bonus = 0.0
        reasons: List[str] = []

        section = metadata.get("section", "")
        if "sections" in filters and section in filters["sections"]:
            bonus += CONFIG.metadata_section_boost
            reasons.append(f"section={section}")

        raw_subcats = metadata.get("subcategories", "")
        if isinstance(raw_subcats, str):
            subcategories = [s.strip() for s in raw_subcats.split(",") if s.strip()]
        else:
            subcategories = raw_subcats
        if "subcategories" in filters and any(
            sc in subcategories for sc in filters["subcategories"]
        ):
            bonus += CONFIG.metadata_subcategory_boost
            reasons.append("subcat_match")

        page_number = metadata.get("page_number", -1)
        if "page" in filters and page_number == filters["page"]:
            bonus += CONFIG.metadata_page_boost
            reasons.append(f"page={page_number}")

        new_dbg = {**dbg, "metadata_boost": bonus, "metadata_reasons": reasons}
        boosted.append((node, rrf + bonus, new_dbg))

    boosted.sort(key=lambda x: -x[1])
    return boosted


demo_q = "What does GOVERN say about GV.SC-07?"
filters = infer_filters_from_query(demo_q)
print(f"Inferred filters from {demo_q!r}: {filters}")

## 12. Cross-encoder reranker




In [ ]:
class Reranker:
    def __init__(self, model_name: str = CONFIG.reranker_model):
        self.log = logging.getLogger("rag.reranker")
        self.backend = None
        self.model = None
        try:
            from flashrank import Ranker, RerankRequest
            self.log.info(f"Initializing FlashRank with model={model_name}")
            self.model = Ranker(model_name=model_name, cache_dir="/tmp/flashrank")
            self.backend = "flashrank"
            self._RerankRequest = RerankRequest
        except Exception as e:
            self.log.warning(f"FlashRank unavailable ({e}); falling back to CrossEncoder.")
            from sentence_transformers import CrossEncoder
            ce_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
            self.log.info(f"Initializing CrossEncoder with model={ce_name}")
            self.model = CrossEncoder(ce_name)
            self.backend = "cross-encoder"

    def rerank(
        self,
        query: str,
        candidates: List[Tuple[Any, float, Dict[str, Any]]],
        top_k: int,
    ) -> List[Tuple[Any, float, Dict[str, Any]]]:
        if not candidates:
            return []
        t0 = time.perf_counter()

        
        def _passage_text(node):
            return node.metadata.get("raw_text") or node.text

        if self.backend == "flashrank":
            passages = [
                {"id": i, "text": _passage_text(n), "meta": {}}
                for i, (n, _, _) in enumerate(candidates)
            ]
            req = self._RerankRequest(query=query, passages=passages)
            results = self.model.rerank(req)
            scored = []
            for r in results:
                idx = r["id"]
                rerank_score = float(r["score"])
                node, fused_score, dbg = candidates[idx]
                
                new_dbg = {**dbg, "fused_score": fused_score, "rerank_score": rerank_score}
                scored.append((node, fused_score, new_dbg))
        else:
            pairs = [(query, _passage_text(n)) for n, _, _ in candidates]
            scores = self.model.predict(pairs)
            scored = []
            for (node, fused_score, dbg), rerank_score in zip(candidates, scores):
                new_dbg = {**dbg, "fused_score": fused_score, "rerank_score": float(rerank_score)}
                scored.append((node, fused_score, new_dbg))

        # Sort by rerank score (this is the whole point of reranking),
        # but the score travelling with the tuple stays the fused score.
        scored.sort(key=lambda x: -x[2]["rerank_score"])
        elapsed = time.perf_counter() - t0
        self.log.info(f"Reranked {len(candidates)} → top {top_k} in {elapsed*1000:.1f}ms ({self.backend})")
        return scored[:top_k]


reranker = Reranker()
log.info(f"Reranker backend: {reranker.backend}")

## 13. Pipeline loop




In [ ]:
class RetrievalResult(BaseModel):
    chunk_id: str
    text: str
    page_number: int
    section: str
    subcategories: List[str]
    source_file: str
    raw_text: str
    document_title: str = "NIST Cybersecurity Framework (CSF) 2.0"
    score: float                         # this is the fused/RRF+boost score
    rerank_score: Optional[float] = None # cross-encoder logit, if reranked
    debug: Dict[str, Any] = Field(default_factory=dict)


def _node_to_result(node, score: float, debug: Dict[str, Any]) -> RetrievalResult:
    m = node.metadata
    raw_subcats = m.get("subcategories", "")
    if isinstance(raw_subcats, str):
        subcategories = [s.strip() for s in raw_subcats.split(",") if s.strip()]
    else:
        subcategories = raw_subcats

    raw_text = m.get("raw_text") or node.text
    return RetrievalResult(
        chunk_id=m["chunk_id"],
        text=raw_text,                          
        score=score,
        rerank_score=debug.get("rerank_score"),
        page_number=int(m.get("page_number", -1)),
        section=m.get("section", "UNKNOWN"),
        subcategories=subcategories,
        source_file=m.get("file_name", ""),
        document_title=m.get("document_title", ""),
        debug=debug,
        raw_text=raw_text,
    )


def hybrid_retrieve(query: str, top_k: int = CONFIG.final_top_k, apply_rerank: bool = True):
    timings: Dict[str, float] = {}
    plog = logging.getLogger("rag.pipeline")

    # Language routing
    t0 = time.perf_counter()
    lang = detect_language(query)
    search_fn = chroma_search_multi if lang == "ar" else chroma_search
    timings["language_detect"] = time.perf_counter() - t0
    plog.info(f"Language: {lang} → {'multilingual' if lang == 'ar' else 'primary'} index")

    # BM25
    t0 = time.perf_counter()
    bm25_results = bm25_retriever.search(query, k=CONFIG.bm25_top_k)
    timings["bm25"] = time.perf_counter() - t0
    plog.info(f"BM25: {len(bm25_results)} hits in {timings['bm25']*1000:.1f}ms")

    t0 = time.perf_counter()
    chroma_results = search_fn(query, k=CONFIG.faiss_top_k)
    timings["chroma"] = time.perf_counter() - t0
    plog.info(f"Chroma: {len(chroma_results)} hits in {timings['chroma']*1000:.1f}ms")

    t0 = time.perf_counter()
    fused = reciprocal_rank_fusion([bm25_results, chroma_results])
    timings["rrf"] = time.perf_counter() - t0
    plog.info(f"RRF: {len(fused)} unique candidates")

    # Metadata filter — operates on the same TextNodes
    t0 = time.perf_counter()
    filters = infer_filters_from_query(query)
    if filters:
        plog.info(f"Filters inferred: {filters}")
    fused = apply_metadata_filters(fused, filters)
    timings["metadata_filter"] = time.perf_counter() - t0

    # Rerank
    t0 = time.perf_counter()
    rr_input = fused[:10]
    if apply_rerank:
        reranked = reranker.rerank(query, rr_input, top_k=top_k)
    else:
        reranked = [(n, s, d) for n, s, d in rr_input[:top_k]]
    timings["rerank"] = time.perf_counter() - t0

    results = [_node_to_result(n, s, d) for n, s, d in reranked]
    timings["total"] = sum(v for k, v in timings.items() if k != "total")
    plog.info(f"Pipeline total: {timings['total']*1000:.1f}ms | {len(results)} results")
    return results, timings


res, t = hybrid_retrieve("How does the framework address supply chain risks?")
print(f"\nTop {len(res)} for 'supply chain risks':")
for r in res:
    print(f"  fused={r.score:.4f}  rerank={r.rerank_score}  page={r.page_number}  section={r.section}")
    print(f"    {r.text[:160]}…")

## 14. Grounded synthesis with gpt4 mini


In [ ]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI

llama_llm = None 
try:
    log.info(f"Loading synthesis model: {CONFIG.rag_model} (OpenAI)")
    _t0 = time.perf_counter()
    llama_llm = OpenAI(
        model=CONFIG.rag_model,
        temperature=0.2,
        max_tokens=1024,
        timeout=60.0,
       
    )
    Settings.llm = llama_llm
    log.info(f"{CONFIG.rag_model} ready in {time.perf_counter()-_t0:.2f}s")
except Exception as e:
    log.error(f"Failed to load OpenAI LLM: {e}")
    llama_llm = None


class SourceCitation(BaseModel):
    chunk_id: str
    page_number: int
    section: str
    snippet: str
    score: float


class StructuredAnswer(BaseModel):
    query: str
    answer: str
    sources: List[SourceCitation]
    confidence: float = Field(..., ge=0.0, le=1.0)
    reasoning: str
    language: str
    cached: bool = False
    timing_ms: Dict[str, float] = Field(default_factory=dict)
    model_used: str = "unknown"
    pipeline_version: str = "3.0.0"


# 
GROUNDED_PROMPT = (
    "You are a cybersecurity expert assistant specialized in the NIST Cybersecurity Framework (CSF) 2.0. "
    "You answer questions based ONLY on the retrieved context excerpts from the official NIST CSF 2.0 publication. "
    "Each context block begins with [N] (page X, SECTION) where SECTION is one of GOVERN, IDENTIFY, PROTECT, DETECT, RESPOND, RECOVER, or GENERAL. "

    "CRITICAL LANGUAGE RULE: Each user message will start with 'RESPOND IN: <language>'. "
    "You MUST respond entirely in that language. Never mix languages. Never switch languages mid-answer. "

    "IMPORTANT RULES: "
    "1. If the retrieved context contains relevant information, use it to provide a helpful, technically accurate answer. "
    "2. Use semantic understanding, not just lexical matching. For example, 'monitor third-party vendors' matches content about 'supplier risks recorded, prioritized, assessed'. "
    "3. Preserve subcategory identifiers EXACTLY as written in the context: GV.SC-07, PR.AA-01, DE.AE-08, etc. Never paraphrase, translate, or alter them. "
    "4. Cite sources inline using bracket numbers from the context, like [1] and [2]. "
    "5. When a question mentions a specific CSF Function or subcategory ID, prioritize excerpts from that area. "
    "6. Combine multiple relevant excerpts into one coherent answer. "
    "7. If the context only partially answers the question, give the supported part and note what is not covered. "

    "ONLY refuse to answer if the context is completely unrelated to the question or no reasonable connection can be made. "
    "When refusing, use the exact REFUSAL PHRASE provided in the user message. "

    "Be helpful, concise, technically precise, and grounded in the cited excerpts."
)


def _build_context(results: List[RetrievalResult]) -> str:
    return "\n\n".join(
        f"[{i}] (page {r.page_number}, {r.section})\n{r.raw_text}"
        for i, r in enumerate(results, start=1)
    )


def synthesize_with_llama(query: str, results: List[RetrievalResult]):
    if not results:
        raise RuntimeError("No retrieval results available.")
    if llama_llm is None:
        raise RuntimeError("LLM synthesis model is unavailable.")

    lang = detect_language(query)
    if lang == "ar":
        lang_label = "Arabic"
        refusal = "هذه المعلومة غير موجودة في المستندات المرفوعة"
    else:
        lang_label = "English"
        refusal = "This information is not available in the uploaded documents"

    user_message = (
        f"RESPOND IN: {lang_label}\n"
        f"REFUSAL PHRASE (use only if you must refuse): {refusal}\n\n"
        f"Context:\n{_build_context(results)}\n\n"
        f"Question:\n{query}\n\n"
        f"Answer (in {lang_label}):"
    )

    from llama_index.core.llms import ChatMessage, MessageRole
    response = llama_llm.chat([
        ChatMessage(role=MessageRole.SYSTEM, content=GROUNDED_PROMPT),
        ChatMessage(role=MessageRole.USER, content=user_message),
    ])
    answer = str(response.message.content).strip()
    if not answer:
        raise RuntimeError("LLM returned empty response.")
    return answer, CONFIG.rag_model

def estimate_confidence(results: List[RetrievalResult]) -> float:
    
    if not results:
        return 0.0

    
    if results[0].rerank_score is not None:
        
        top = results[0].rerank_score
        conf = 1.0 / (1.0 + np.exp(-top))
        if len(results) >= 2 and results[1].rerank_score is not None:
            gap = results[0].rerank_score - results[1].rerank_score
            
            conf = min(1.0, conf + max(0.0, gap) * 0.02)
    else:
       
        top = results[0].score
        conf = float(np.clip(top / 0.05, 0.0, 1.0))
        if len(results) >= 2:
            gap = results[0].score - results[1].score
            conf = float(np.clip(conf + max(0.0, gap) * 5.0, 0.0, 1.0))

    return float(conf)

## 15. Top-level `ask()`


In [ ]:
def ask(query: str, top_k: int = CONFIG.final_top_k, use_cache: bool = True) -> StructuredAnswer:
    """Cache → BM25 → Chroma → RRF → metadata filter → rerank → Llama → structured output."""
    pipe_log = logging.getLogger("rag.ask")
    pipe_log.info("━" * 70)
    pipe_log.info(f"NEW QUERY | {query!r}")
    overall_t0 = time.perf_counter()
    timings: Dict[str, float] = {}

   
    if use_cache:
        t0 = time.perf_counter()
        cached = semantic_cache.lookup(query)
        timings["cache_lookup"] = (time.perf_counter() - t0) * 1000
        if cached is not None:
            timings["total"] = (time.perf_counter() - overall_t0) * 1000
            return StructuredAnswer(**{
                **{k: v for k, v in cached.items() if not k.startswith("_")},
                "cached": True,
                "timing_ms": timings,
            })

    
    results, retrieval_timings = hybrid_retrieve(query, top_k=top_k)
    for k, v in retrieval_timings.items():
        timings[f"retrieval.{k}"] = v * 1000

    
    t0 = time.perf_counter()
    answer_text, model_used = synthesize_with_llama(query, results)
    timings["synthesis"] = (time.perf_counter() - t0) * 1000

   
    sources = [
        SourceCitation(
            chunk_id=r.chunk_id, page_number=r.page_number, section=r.section,
            snippet=r.text[:280] + ("…" if len(r.text) > 280 else ""), score=r.score,
        ) for r in results
    ]
    confidence = estimate_confidence(results)
    reasoning = (
        f"Hybrid BM25 + Chroma retrieval over {chroma_collection.count()} nodes, "
        f"RRF-fused, metadata-boosted, reranked with {reranker.backend}. "
        f"Synthesis: {model_used}."
    )
    timings["total"] = (time.perf_counter() - overall_t0) * 1000

    answer = StructuredAnswer(
        query=query, answer=answer_text, sources=sources,
        confidence=confidence, reasoning=reasoning,
        language=detect_language(query), cached=False,
        timing_ms=timings, model_used=model_used,
    )
    if use_cache:
        semantic_cache.store(query, answer.model_dump())

    pipe_log.info(f"Done: confidence={confidence:.3f}  total={timings['total']:.1f}ms  "
                  f"sources={len(sources)}  model={model_used}")
    return answer

In [ ]:
def pretty_print(answer: StructuredAnswer):
    print("\n" + "═" * 80)
    print(f"QUERY ({answer.language}): {answer.query}")
    print("═" * 80)
    print(f"\nANSWER (confidence={answer.confidence:.3f}, cached={answer.cached}, model={answer.model_used}):")
    print(f"  {answer.answer}")
    print(f"\nREASONING:\n  {answer.reasoning}")
    print(f"\nSOURCES ({len(answer.sources)}):")
    for i, s in enumerate(answer.sources, 1):
        print(f"  [{i}] {s.chunk_id} | page {s.page_number} | {s.section} | score={s.score:.4f}")
        print(f'      "{s.snippet[:180]}…"')
    if answer.timing_ms:
        ts = " | ".join(f"{k}={v:.1f}ms" for k, v in answer.timing_ms.items() if v > 0.1)
        print(f"\nTIMING:  {ts}")
    print("═" * 80)

## 16. Demos


In [ ]:
# DEMO 1: Identifier-anchored query — BM25 strength
print("\n\n>>> DEMO 1: Verbatim subcategory identifier (BM25 strength) <<<")
ans = ask("What is GV.SC-07?")
pretty_print(ans)

In [ ]:
# DEMO 2: Conceptual paraphrase — semantic strength
print("\n\n>>> DEMO 2: Conceptual paraphrase (semantic strength) <<<")
ans = ask("How should an organization keep tabs on third-party software vendors?")
pretty_print(ans)

In [ ]:
# DEMO 3: Function-scoped query — metadata boost activates (now mild, not dominant)
print("\n\n>>> DEMO 3: Function-scoped query (metadata boost activates) <<<")
ans = ask("Under GOVERN, how is risk appetite communicated?")
pretty_print(ans)

In [ ]:
# DEMO 4: Repeat the same query → cache HIT
print("\n\n>>> DEMO 4: Repeat exact query — should HIT cache <<<")
ans = ask("What is GV.SC-07?")
pretty_print(ans)
print(f"\nCache stats: {semantic_cache.stats()}")

In [ ]:
# DEMO 5: Paraphrase — semantic cache should still HIT
print("\n\n>>> DEMO 5: Paraphrase of an earlier query — semantic cache test <<<")
ans = ask("Tell me about GV.SC-07.")
pretty_print(ans)
print(f"\nCache stats: {semantic_cache.stats()}")

In [ ]:
# DEMO 6: Arabic query — multilingual index AND multilingual cache embedder
print("\n\n>>> DEMO 6: Arabic query — multilingual retrieval <<<")
arabic_query = "ما هي وظائف إطار الأمن السيبراني الست؟"
ans = ask(arabic_query)
pretty_print(ans)

In [ ]:
# DEMO 7: Another Arabic query
print("\n\n>>> DEMO 7: Another Arabic query — supply chain in Arabic <<<")
arabic_query = "كيف يدير الإطار مخاطر سلسلة التوريد؟"
ans = ask(arabic_query)
pretty_print(ans)